# Data audit

Validate parsing, timestamp ordering, missing-value rates, red-team matching, positive coverage across splits, and DNS pair coverage (README sections 3, 4, 11).

In [ ]:
import json
import sys
from pathlib import Path

import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

CONFIG = yaml.safe_load((ROOT / "configs" / "experiment.yaml").read_text())

In [ ]:
from knightglass.download import fetch_raw

RAW = fetch_raw(CONFIG, root=ROOT)
RAW

## Ingest to day-partitioned Parquet

Streams each archive in blocks, assigns stable row IDs, and checks that timestamps never decrease. Re-running requires deleting the output folder first. The full authentication archive took about 110 minutes in the first full run (1,051,430,459 rows, about 8.5 GB of Parquet).

In [ ]:
from knightglass.ingestion import AUTH_COLUMNS, DNS_COLUMNS, stream_to_parquet

INTERIM = ROOT / CONFIG["data"]["interim_dir"]

manifests = {}
for name, columns in [("dns", DNS_COLUMNS), ("auth", AUTH_COLUMNS)]:
    out = INTERIM / name
    if (out / "_manifest.json").exists():
        manifests[name] = json.loads((out / "_manifest.json").read_text())
    else:
        manifests[name] = stream_to_parquet(RAW[name], columns, out)

In [ ]:
import pandas as pd

pd.DataFrame({name: m["rows_per_day"] for name, m in manifests.items()}).fillna(0).astype(int)

## Red-team matching audit

Keys match on time, source user, source computer and destination computer (README section 4).

In [ ]:
from knightglass.ingestion import match_redteam

matches, audit = match_redteam(INTERIM / "auth", RAW["redteam"], INTERIM / "labels")
{k: v for k, v in audit.items() if k not in ("keys_by_day", "unmatched_examples")}

In [ ]:
pd.DataFrame(audit["keys_by_day"]).T

## Possible red-team rows

Unlabeled rows that meet the section 4.2 rule, plus the documented exceptions in `configs/experiment.yaml`.

In [ ]:
import pyarrow.parquet as pq

from knightglass.ingestion import possible_redteam

LABELS = INTERIM / "labels"
keys = pq.read_table(LABELS / "redteam_keys.parquet")
possible = possible_redteam(INTERIM / "auth", keys, matches,
                            CONFIG["labels"]["possible_exceptions"], LABELS)
possible.group_by("reason").aggregate([("row_id", "count")]).to_pandas()

## Known-positive coverage across the splits

Only matched keys count toward recall (README section 4.3). A split with no matched keys cannot measure recall.

In [ ]:
from knightglass.evaluation import split_coverage

coverage = split_coverage(keys, matches, possible, CONFIG["splits"])
pd.DataFrame(coverage).T